# Brain Tumor MRI Classification: CNN vs DCFormer-lite

4-class classification (glioma / meningioma / pituitary / no tumor) on the Kaggle brain tumor MRI dataset.

What I'm doing here:
- check the dataset for duplicate images leaking between train and test
- build a clean 70/15/15 split
- train a baseline CNN and a DCFormer-style model (decomposed depthwise convs)
- compare accuracy, AUC, FLOPs and params
- Grad-CAM to see where each model looks
- quick 3D compute comparison at the end

## 1. Setup

In [ ]:
import os
import random
import json
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 2. Download dataset

Kaggle: sartajbhuvaji/brain-tumor-classification-mri. Needs Kaggle username + API key the first time.

In [ ]:
if os.environ.get("DATA_DIR"):
    path = os.environ["DATA_DIR"]
else:
    import kagglehub
    kagglehub.login()
    path = kagglehub.dataset_download("sartajbhuvaji/brain-tumor-classification-mri")

print(path)

train_dir = os.path.join(path, "Training")
test_dir  = os.path.join(path, "Testing")
print(os.listdir(path))
print(os.listdir(train_dir))

## 3. Explore

Folder name = label.

In [ ]:
CLASSES = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]

def collect_images(root_dir, split_name):
    records = []
    for label_idx, class_name in enumerate(CLASSES):
        class_dir = os.path.join(root_dir, class_name)
        for fname in sorted(os.listdir(class_dir)):
            records.append({
                "path": os.path.join(class_dir, fname),
                "label": label_idx,
                "split": split_name,
            })
    return records

train_records = collect_images(train_dir, "train")
test_records  = collect_images(test_dir,  "test")
all_records   = train_records + test_records

print("train:", len(train_records))
print("test: ", len(test_records))
print("total:", len(all_records))

for split_name, recs in [("train", train_records), ("test", test_records)]:
    counts = Counter(r["label"] for r in recs)
    print(f"\n{split_name}")
    for idx, name in enumerate(CLASSES):
        print(f"   {name:18s} {counts[idx]}")

In [ ]:
# one image per class
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for idx, class_name in enumerate(CLASSES):
    rec = next(r for r in train_records if r["label"] == idx)
    img = Image.open(rec["path"])
    axes[idx].imshow(img, cmap="gray")
    axes[idx].set_title(f"{class_name}\n{img.size[0]}x{img.size[1]}")
    axes[idx].axis("off")
plt.tight_layout()
plt.show()

## 4. Duplicate check (data leakage)

Using dHash to fingerprint every image. Same fingerprint = same scan, even if resized / re-saved.
Then check how many test images also exist in train.

In [ ]:
def image_fingerprint(img_path, hash_size=8):
    img = Image.open(img_path).convert("L")
    img = img.resize((hash_size + 1, hash_size), Image.LANCZOS)
    px = np.asarray(img, dtype=np.int16)
    diff = px[:, 1:] > px[:, :-1]
    return "".join("1" if b else "0" for b in diff.flatten())

for rec in all_records:
    rec["fp"] = image_fingerprint(rec["path"])

groups = defaultdict(list)
for rec in all_records:
    groups[rec["fp"]].append(rec)

for gid, (fp, recs) in enumerate(groups.items()):
    for rec in recs:
        rec["group"] = gid

n_unique = len(groups)
n_dup_images = sum(len(r) for r in groups.values() if len(r) > 1)
print("unique images:", n_unique)
print("images in a duplicate group:", n_dup_images)

# how much of the official test set is also in train?
train_fps = {r["fp"] for r in train_records}
leaked = [r for r in test_records if r["fp"] in train_fps]
leak_pct = 100 * len(leaked) / len(test_records)
print(f"\ntest images duplicated in train: {len(leaked)} / {len(test_records)} = {leak_pct:.1f}%")

conflicts = sum(1 for recs in groups.values() if len({r['label'] for r in recs}) > 1)
print("duplicate groups with conflicting labels:", conflicts)

In [ ]:
# show one leaked pair
if leaked:
    test_rec = leaked[0]
    train_rec = next(r for r in train_records if r["fp"] == test_rec["fp"])
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(Image.open(train_rec["path"]), cmap="gray"); axes[0].set_title("train")
    axes[1].imshow(Image.open(test_rec["path"]),  cmap="gray"); axes[1].set_title("test")
    for ax in axes: ax.axis("off")
    plt.show()
    print(os.path.basename(train_rec["path"]), "|", os.path.basename(test_rec["path"]))

## 5. Clean split

Ignore the official split. Split by duplicate group (not by image), stratified by class, 70/15/15.

In [ ]:
def grouped_stratified_split(records, train_frac=0.70, val_frac=0.15, seed=SEED):
    rng = random.Random(seed)

    group_label = {}
    for gid in {r["group"] for r in records}:
        labels = [r["label"] for r in records if r["group"] == gid]
        group_label[gid] = Counter(labels).most_common(1)[0][0]

    split_of_group = {}
    for cls in range(len(CLASSES)):
        gids = [g for g, l in group_label.items() if l == cls]
        rng.shuffle(gids)
        n = len(gids)
        n_train = int(n * train_frac)
        n_val   = int(n * val_frac)
        for i, g in enumerate(gids):
            if i < n_train:            split_of_group[g] = "train"
            elif i < n_train + n_val:  split_of_group[g] = "val"
            else:                      split_of_group[g] = "test"

    for rec in records:
        rec["my_split"] = split_of_group[rec["group"]]
    return records

all_records = grouped_stratified_split(all_records)

# keep one copy per group
seen = set()
clean_records = []
for rec in all_records:
    if rec["group"] not in seen:
        seen.add(rec["group"])
        clean_records.append(rec)

splits = {name: [r for r in clean_records if r["my_split"] == name] for name in ["train", "val", "test"]}
for name, recs in splits.items():
    print(f"{name:5s}: {len(recs):4d}  per class:",
          [sum(1 for r in recs if r['label'] == c) for c in range(len(CLASSES))])

# sanity check
fps_by_split = {name: {r["fp"] for r in recs} for name, recs in splits.items()}
assert not (fps_by_split["train"] & fps_by_split["test"])
assert not (fps_by_split["train"] & fps_by_split["val"])
print("\nno overlap between splits")

## 6. Dataset + DataLoader

Grayscale, resize to 128x128, normalize to [-1, 1]. Random horizontal flip for training.

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

class MRIDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records = records
        self.augment = augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        rec = self.records[i]
        img = Image.open(rec["path"]).convert("L").resize((IMG_SIZE, IMG_SIZE))
        x = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0)
        x = (x - 0.5) / 0.5
        x = x.unsqueeze(0)
        if self.augment and random.random() < 0.5:
            x = torch.flip(x, dims=[2])
        return x, rec["label"]

train_ds = MRIDataset(splits["train"], augment=True)
val_ds   = MRIDataset(splits["val"])
test_ds  = MRIDataset(splits["test"])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape)

## 7. Baseline CNN

4 x (Conv3x3 -> BN -> ReLU -> MaxPool), global avg pool, linear.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        chans = [1, 32, 64, 128, 256]
        blocks = []
        for cin, cout in zip(chans[:-1], chans[1:]):
            blocks += [
                nn.Conv2d(cin, cout, kernel_size=3, padding=1),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ]
        self.features = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(chans[-1], num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

cnn = BaselineCNN().to(device)
print(cnn)
print("params:", sum(p.numel() for p in cnn.parameters()))

## 8. DCFormer-lite

Each block: depthwise 1xk -> depthwise kx1 -> BN -> 1x1 -> GELU -> 1x1, with a residual.
Depthwise + decomposed kernels = way fewer FLOPs than a full 3x3 conv.

In [ ]:
class DecomposedDWBlock(nn.Module):
    def __init__(self, dim, k=7):
        super().__init__()
        self.dw_h = nn.Conv2d(dim, dim, kernel_size=(1, k), padding=(0, k // 2), groups=dim)
        self.dw_v = nn.Conv2d(dim, dim, kernel_size=(k, 1), padding=(k // 2, 0), groups=dim)
        self.norm = nn.BatchNorm2d(dim)
        self.pw1 = nn.Conv2d(dim, dim * 2, kernel_size=1)
        self.act = nn.GELU()
        self.pw2 = nn.Conv2d(dim * 2, dim, kernel_size=1)

    def forward(self, x):
        shortcut = x
        x = self.dw_h(x)
        x = self.dw_v(x)
        x = self.norm(x)
        x = self.pw2(self.act(self.pw1(x)))
        return x + shortcut

class DCFormerLite(nn.Module):
    def __init__(self, num_classes=4, dims=(24, 48, 96, 192), depth=1):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, dims[0], kernel_size=4, stride=4),
            nn.BatchNorm2d(dims[0]),
        )
        stages = []
        for i, d in enumerate(dims):
            if i > 0:
                stages.append(nn.Sequential(
                    nn.BatchNorm2d(dims[i - 1]),
                    nn.Conv2d(dims[i - 1], d, kernel_size=2, stride=2)))
            stages += [DecomposedDWBlock(d) for _ in range(depth)]
        self.stages = nn.Sequential(*stages)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(dims[-1], num_classes)

    def forward(self, x):
        x = self.stages(self.stem(x))
        return self.classifier(self.pool(x).flatten(1))

dcf = DCFormerLite().to(device)
print("params:", sum(p.numel() for p in dcf.parameters()))

## 9. FLOPs

Forward hooks on every conv / linear layer, count multiply-adds for one image.

In [ ]:
def count_flops(model, input_shape):
    flops = []

    def conv_hook(module, inp, out):
        k = int(np.prod(module.kernel_size))
        cin_per_group = module.in_channels // module.groups
        flops.append(out.numel() * k * cin_per_group)

    def linear_hook(module, inp, out):
        flops.append(module.in_features * module.out_features)

    handles = []
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Conv3d)):
            handles.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear):
            handles.append(m.register_forward_hook(linear_hook))

    model.eval()
    with torch.no_grad():
        model(torch.zeros(1, *input_shape, device=next(model.parameters()).device))
    for h in handles:
        h.remove()
    return sum(flops)

cnn_flops = count_flops(cnn, (1, IMG_SIZE, IMG_SIZE))
dcf_flops = count_flops(dcf, (1, IMG_SIZE, IMG_SIZE))
print(f"CNN:      {cnn_flops/1e6:8.1f} MFLOPs")
print(f"DCFormer: {dcf_flops/1e6:8.1f} MFLOPs")
print(f"ratio: {cnn_flops/dcf_flops:.1f}x")

## 10. Training

AdamW + cosine LR, cross-entropy. Keep the best epoch by val accuracy.

In [ ]:
EPOCHS = int(os.environ.get("EPOCHS", 15))

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.set_grad_enabled(training):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(xb)
            all_preds.append(logits.argmax(1).cpu())
            all_labels.append(yb.cpu())
    preds, labels = torch.cat(all_preds), torch.cat(all_labels)
    acc = (preds == labels).float().mean().item()
    return total_loss / len(loader.dataset), acc

def train_model(model, name):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best_acc, best_state, history = 0.0, None, []
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader)
        scheduler.step()
        history.append({"epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc,
                        "val_loss": va_loss, "val_acc": va_acc})
        if va_acc > best_acc:
            best_acc = va_acc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        print(f"[{name}] epoch {epoch:2d} | train loss {tr_loss:.3f} acc {tr_acc:.3f} "
              f"| val loss {va_loss:.3f} acc {va_acc:.3f}")
    model.load_state_dict(best_state)
    torch.save(best_state, f"{name}_best.pt")
    print(f"[{name}] best val acc: {best_acc:.3f}")
    return history

cnn_history = train_model(cnn, "cnn")

In [ ]:
dcf_history = train_model(dcf, "dcformer")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for hist, label in [(cnn_history, "CNN"), (dcf_history, "DCFormer-lite")]:
    axes[0].plot([h["val_loss"] for h in hist], label=label)
    axes[1].plot([h["val_acc"]  for h in hist], label=label)
axes[0].set_title("val loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("val acc");  axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## 11. Test set evaluation

In [ ]:
def evaluate(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            probs.append(F.softmax(logits, dim=1).cpu())
            labels.append(yb)
    probs, labels = torch.cat(probs).numpy(), torch.cat(labels).numpy()
    preds = probs.argmax(1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "auc": roc_auc_score(labels, probs, multi_class="ovr"),
        "confusion": confusion_matrix(labels, preds, labels=list(range(len(CLASSES)))),
    }

results = {}
for model, name, flops in [(cnn, "Baseline CNN", cnn_flops), (dcf, "DCFormer-lite", dcf_flops)]:
    r = evaluate(model, test_loader)
    r["mflops"] = flops / 1e6
    r["params"] = sum(p.numel() for p in model.parameters())
    results[name] = r
    print(f"{name:14s} | acc {r['accuracy']*100:5.1f}% | AUC {r['auc']:.3f} "
          f"| {r['mflops']:7.1f} MFLOPs | {r['params']/1e6:.2f}M params")

In [ ]:
# confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, r) in zip(axes, results.items()):
    cm = r["confusion"]
    ax.imshow(cm, cmap="Blues")
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(4)); ax.set_xticklabels([c.replace("_tumor", "") for c in CLASSES], rotation=30)
    ax.set_yticks(range(4)); ax.set_yticklabels([c.replace("_tumor", "") for c in CLASSES])
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(name)
plt.tight_layout(); plt.show()

## 12. Grad-CAM

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_act)
        target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, inp, out):   self.activations = out.detach()
    def _save_grad(self, module, gin, gout): self.gradients = gout[0].detach()

    def __call__(self, x):
        self.model.eval()
        x = x.to(device).unsqueeze(0)
        logits = self.model(x)
        cls = logits.argmax(1).item()
        self.model.zero_grad()
        logits[0, cls].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        cam = cam[0, 0].cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, cls

cam_cnn = GradCAM(cnn, cnn.features[-4])
cam_dcf = GradCAM(dcf, dcf.stages[-1])

n_show = 4
fig, axes = plt.subplots(n_show, 3, figsize=(10, 3.2 * n_show))
for row in range(n_show):
    x, y = test_ds[row * (len(test_ds) // n_show)]
    img = x[0].numpy()
    axes[row, 0].imshow(img, cmap="gray"); axes[row, 0].set_title(f"true: {CLASSES[y]}")
    for col, (cam_obj, name) in enumerate([(cam_cnn, "CNN"), (cam_dcf, "DCFormer-lite")], start=1):
        cam, pred = cam_obj(x)
        axes[row, col].imshow(img, cmap="gray")
        axes[row, col].imshow(cam, cmap="jet", alpha=0.45)
        axes[row, col].set_title(f"{name} -> {CLASSES[pred]}")
    for ax in axes[row]: ax.axis("off")
plt.tight_layout(); plt.show()

## 13. 3D: full 3x3x3 conv vs decomposed 1D depthwise convs

Same FLOP counter on a dummy 32-channel 64x64x64 volume.

In [ ]:
class Full3DBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv = nn.Conv3d(c, c, kernel_size=3, padding=1)
    def forward(self, x): return self.conv(x)

class Decomposed3DBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.d = nn.Conv3d(c, c, kernel_size=(3, 1, 1), padding=(1, 0, 0), groups=c)
        self.h = nn.Conv3d(c, c, kernel_size=(1, 3, 1), padding=(0, 1, 0), groups=c)
        self.w = nn.Conv3d(c, c, kernel_size=(1, 1, 3), padding=(0, 0, 1), groups=c)
        self.pw = nn.Conv3d(c, c, kernel_size=1)
    def forward(self, x): return self.pw(self.w(self.h(self.d(x))))

C, D = 32, 64
full_flops = count_flops(Full3DBlock(C).to(device),       (C, D, D, D))
deco_flops = count_flops(Decomposed3DBlock(C).to(device), (C, D, D, D))
saving = 100 * (1 - deco_flops / full_flops)
print(f"full 3x3x3:  {full_flops/1e9:6.2f} GFLOPs")
print(f"decomposed:  {deco_flops/1e9:6.2f} GFLOPs")
print(f"saving: {saving:.0f}%")

## 14. Save results

In [ ]:
summary = {
    "dataset": {
        "official_train": len(train_records),
        "official_test": len(test_records),
        "unique_images": n_unique,
        "test_images_duplicated_in_train_pct": round(leak_pct, 1),
        "clean_split": {k: len(v) for k, v in splits.items()},
    },
    "models": {name: {"accuracy_pct": round(r["accuracy"] * 100, 1),
                      "auc": round(r["auc"], 3),
                      "mflops": round(r["mflops"], 1),
                      "params": r["params"]} for name, r in results.items()},
    "flops_ratio_cnn_over_dcformer": round(cnn_flops / dcf_flops, 1),
    "3d_decomposition_compute_saving_pct": round(saving, 0),
}
with open("results.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))